In [1]:
!pip install streamlink ultralytics
!apt-get install ffmpeg -y|

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 11.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 106.9 MB/s eta 0:00:0000:01
/bin/bash: -c: line 2: syntax error: unexpected end of file


In [ ]:
import cv2
import time
import os
import streamlink
import numpy as np
import threading
import requests
import urllib3
from ultralytics import YOLO
from datetime import datetime
from collections import deque

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- CONFIG ---
MODEL = YOLO('yolo11m.pt')
DEVICE = 'cuda:0'
STATIC_ROOT = os.path.abspath("Static_Storage").replace("\\", "/")

EXIT_THRESHOLD = 8.0   
PRE_ROLL_SECONDS = 5   
CLIP_DURATION = 30     
FPS = 10.0  

# --- OPENREMOTE ---
OR_BASE_URL = "https://109.176.197.144"
OR_REALM = "aiprojects"
OR_CLIENT_ID = "aiprojects"
OR_SECRET = "nV3eMyOSoIHCgRtqII0qiueDvgOCM670"

tracking_db = {0: {}, 1: {}}
part_counters = {0: 1, 1: 1} # Part counters for each cam
zone_snapshot_paths = {0: "", 1: ""}
gpu_lock = threading.Lock()
stop_now = False

CAMS_DATA = {
    0: {"name": "Dublin", "url": "https://www.earthcam.com/world/ireland/dublin/?cam=templebar", "asset_id": "7d2UcWwY76HUjMGYN4RgvV", "res": (955, 542), "zone": np.array([[134, 339], [254, 332], [319, 403], [132, 448]], dtype=np.int32)},
    1: {"name": "NYC", "url": "https://www.earthcam.com/usa/newyork/timessquare/?cam=tsrobo1", "asset_id": "5u4zFNrPIP8GGNdhsxt2HU", "res": (955, 537), "zone": np.array([[120, 73], [646, 73], [646, 502], [120, 502]], dtype=np.int32)}
}

def sync_to_openremote(data_dict, asset_id, cam_name):
    def run():
        try:
            r = requests.post(f"{OR_BASE_URL}/auth/realms/{OR_REALM}/protocol/openid-connect/token", 
                              data={"grant_type": "client_credentials", "client_id": OR_CLIENT_ID, "client_secret": OR_SECRET}, 
                              verify=False, timeout=10)
            token = r.json().get("access_token")
            if token:
                clean_payload = {str(k): str(v) for k, v in data_dict.items() if not k.startswith('_')}
                payload = [{"ref": {"id": asset_id, "name": "Data"}, "value": clean_payload}]
                res = requests.put(f"{OR_BASE_URL}/api/{OR_REALM}/asset/attributes", 
                                   json=payload, headers={"Authorization": f"Bearer {token}"}, verify=False, timeout=10)
                if res.status_code in [200, 204]:
                    print(f" [SUCCESS] {cam_name} Sync -> ID: {clean_payload.get('person_id')} | {clean_payload.get('video_name')}")
        except Exception as e:
            print(f" [!] Sync Error: {e}")
    threading.Thread(target=run, daemon=True).start()

def process_camera(cam_id):
    global stop_now, tracking_db, part_counters
    cfg = CAMS_DATA[cam_id]
    name, tw, th, zone_pts, asset_id = cfg["name"], cfg["res"][0], cfg["res"][1], cfg["zone"], cfg["asset_id"]
    
    paths = {k: os.path.join(STATIC_ROOT, name, v).replace("\\", "/") for k,v in {"vids":"Videos", "caps":"Captures", "zones":"ZoneSnapshots"}.items()}
    for p in paths.values(): os.makedirs(p, exist_ok=True)
    
    frame_buffer = deque(maxlen=int(FPS * PRE_ROLL_SECONDS))
    writer = None
    is_recording = False
    rec_start_time = 0
    active_video_path = ""
    active_video_name = ""
    zone_saved = False

    while not stop_now:
        try:
            streams = streamlink.Streamlink().streams(cfg["url"])
            cap = cv2.VideoCapture(streams['720p'].url if '720p' in streams else streams['best'].url)
            print(f"[*] {name} Monitoring Started.")
        except: time.sleep(5); continue
        
        while not stop_now and cap.isOpened():
            ret, frame = cap.read()
            if not ret: break 
            frame = cv2.resize(frame, (tw, th))
            raw_frame, curr_time = frame.copy(), time.time()
            
            with gpu_lock:
                results = MODEL.track(frame, persist=True, imgsz=640, conf=0.25, tracker="bytetrack.yaml", verbose=False, device=DEVICE, classes=[0])
            
            annotated_frame = frame.copy()
            cv2.polylines(annotated_frame, [zone_pts], True, (0, 255, 255), 2)
            
            if not zone_saved:
                z_path = f"{paths['zones']}/Zone_{name}.jpg"
                cv2.imwrite(z_path, annotated_frame)
                zone_snapshot_paths[cam_id] = z_path
                zone_saved = True

            in_zone_ids_now = []
            if results[0].boxes.id is not None:
                boxes, ids = results[0].boxes.xyxy.cpu().numpy(), results[0].boxes.id.cpu().numpy().astype(int)
                for box, obj_id in zip(boxes, ids):
                    x1, y1, x2, y2 = map(int, box)
                    if cv2.pointPolygonTest(zone_pts, (int((x1+x2)/2), y2), False) >= 0:
                        in_zone_ids_now.append(obj_id)
                        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                        
                        if obj_id not in tracking_db[cam_id]:
                            print(f" [ALERT] {name} Detection: ID {obj_id}")
                            cp = f"{paths['caps']}/I_{obj_id}_{int(curr_time)}.jpg"
                            cv2.imwrite(cp, frame[y1:y2, x1:x2])
                            
                            # Initial Data
                            tracking_db[cam_id][obj_id] = {
                                "person_id": obj_id, "camera": name, 
                                "time_in": datetime.now().strftime("%H:%M:%S"),
                                "intruder_img_url": cp, "_last_seen": curr_time,
                                "video_url": "Recording..."
                            }
                            
                            if not is_recording:
                                is_recording, rec_start_time = True, curr_time
                                active_video_name = f"Part_{part_counters[cam_id]}"
                                active_video_path = f"{paths['vids']}/{name}_{active_video_name}.avi"
                                writer = cv2.VideoWriter(active_video_path, cv2.VideoWriter_fourcc(*'XVID'), FPS, (tw * 2, th))
                                for f in frame_buffer: writer.write(f)
                        else:
                            tracking_db[cam_id][obj_id]["_last_seen"] = curr_time

            combined_frame = np.hstack((raw_frame, annotated_frame))
            frame_buffer.append(combined_frame)

            if is_recording and writer:
                writer.write(combined_frame)
                if (curr_time - rec_start_time) >= CLIP_DURATION:
                    is_recording = False
                    writer.release()
                    writer = None
                    print(f" [VIDEO] {name} {active_video_name} Saved.")
                    
                    # Sabhi active intruders ko yeh Part assign kar do
                    for sid in tracking_db[cam_id]:
                        if tracking_db[cam_id][sid]["video_url"] == "Recording...":
                            tracking_db[cam_id][sid]["video_url"] = active_video_path
                    
                    part_counters[cam_id] += 1 # Increment Part for next clip

            # EXIT LOGIC - Time Out + Part Name + URL
            for sid in list(tracking_db[cam_id].keys()):
                if sid not in in_zone_ids_now and (curr_time - tracking_db[cam_id][sid]["_last_seen"] > EXIT_THRESHOLD):
                    print(f" [EXIT] {name}: ID {sid} Out. Sending Part Data...")
                    tracking_db[cam_id][sid].update({
                        "time_out": datetime.now().strftime("%H:%M:%S"),
                        "zone_snap_url": zone_snapshot_paths[cam_id]
                    })
                    sync_to_openremote(tracking_db[cam_id][sid], asset_id, name)
                    del tracking_db[cam_id][sid]

        cap.release()
        if writer: writer.release()

if __name__ == "__main__":
    print("\n" + "="*40)
    print(" SYSTEM ACTIVE - RECORDING PARTS ")
    print("="*40 + "\n")
    for cid in CAMS_DATA.keys(): threading.Thread(target=process_camera, args=(cid,), daemon=True).start()
    try:
        while True: time.sleep(1)
    except KeyboardInterrupt: stop_now = True

# **FETCH DATA FROM OPENREMOTE**

In [ ]:
import cv2
import time
import os
import streamlink
import numpy as np
import threading
import requests
import urllib3
import json
from ultralytics import YOLO
from datetime import datetime

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- CONFIG ---
MODEL = YOLO('yolo11m.pt')
DEVICE = 'cuda:0'
STATIC_ROOT = os.path.abspath("Static_Storage").replace("\\", "/")

EXIT_THRESHOLD = 8.0   
CLIP_DURATION = 30     
FPS = 10.0  

OR_BASE_URL = "https://109.176.197.144"
OR_REALM = "aiprojects"
OR_CLIENT_ID = "aiprojects"
OR_SECRET = "nV3eMyOSoIHCgRtqII0qiueDvgOCM670"

tracking_db = {0: {}, 1: {}}
last_known_coords = {0: None, 1: None}
part_counters = {0: 1, 1: 1}
gpu_lock = threading.Lock()
stop_now = False

CAMS_DATA = {
    0: {"name": "Dublin", "url": "https://www.earthcam.com/world/ireland/dublin/?cam=templebar", "asset_id": "7d2UcWwY76HUjMGYN4RgvV", "res": (955, 542)},
    1: {"name": "NYC", "url": "https://www.earthcam.com/usa/newyork/timessquare/?cam=tsrobo1", "asset_id": "5u4zFNrPIP8GGNdhsxt2HU", "res": (955, 537)}
}

# --- HELPERS ---

def get_or_token():
    try:
        r = requests.post(f"{OR_BASE_URL}/auth/realms/{OR_REALM}/protocol/openid-connect/token", 
                          data={"grant_type": "client_credentials", "client_id": OR_CLIENT_ID, "client_secret": OR_SECRET}, 
                          verify=False, timeout=5)
        return r.json().get("access_token")
    except: return None

def sync_to_openremote(data_dict, asset_id, cam_name):
    def run():
        token = get_or_token()
        if token:
            try:
                payload_val = {k: v for k, v in data_dict.items() if not k.startswith('_')}
                payload = [{"ref": {"id": asset_id, "name": "Data"}, "value": payload_val}]
                requests.put(f"{OR_BASE_URL}/api/{OR_REALM}/asset/attributes", 
                             json=payload, headers={"Authorization": f"Bearer {token}"}, verify=False, timeout=5)
            except: pass
    threading.Thread(target=run, daemon=True).start()

def fetch_polygon_zone(cam_id):
    asset_id = CAMS_DATA[cam_id]["asset_id"]
    token = get_or_token()
    if not token: return None, None
    try:
        res = requests.get(f"{OR_BASE_URL}/api/{OR_REALM}/asset/{asset_id}", 
                           headers={"Authorization": f"Bearer {token}"}, verify=False, timeout=5)
        attrs = res.json().get("attributes", {})
        c = attrs.get("Coordinates") or attrs.get("coordinates")
        if c and c.get("value"):
            val = c.get("value")
            data = json.loads(val) if isinstance(val, str) else val
            if isinstance(data, dict):
                sx, sy, ex, ey = int(data['startX']), int(data['startY']), int(data['endX']), int(data['endY'])
                pts = np.array([[sx, sy], [ex, sy], [ex, ey], [sx, ey]], dtype=np.int32)
                return pts, [[sx, sy], [ex, sy], [ex, ey], [sx, ey]]
            elif isinstance(data, list):
                pts = np.array(data, dtype=np.int32)
                return pts, data
    except: pass
    return None, None

# --- ENGINE ---

def process_camera(cam_id):
    global stop_now, tracking_db, part_counters, last_known_coords
    cfg = CAMS_DATA[cam_id]
    name, tw, th, asset_id = cfg["name"], cfg["res"][0], cfg["res"][1], cfg["asset_id"]
    
    vids_p = os.path.join(STATIC_ROOT, name, "Videos").replace("\\", "/")
    caps_p = os.path.join(STATIC_ROOT, name, "Captures").replace("\\", "/")
    zone_h_p = os.path.join(STATIC_ROOT, name, "Zone_History").replace("\\", "/")
    for p in [vids_p, caps_p, zone_h_p]: os.makedirs(p, exist_ok=True)

    current_zone_snap_path = ""

    while not stop_now:
        zone_pts, zone_list = fetch_polygon_zone(cam_id)
        if zone_pts is None:
            time.sleep(10); continue

        try:
            streams = streamlink.Streamlink().streams(cfg["url"])
            cap = cv2.VideoCapture(streams['best'].url, cv2.CAP_FFMPEG)
            
            if str(zone_list) != str(last_known_coords[cam_id]):
                ret, f_snap = cap.read()
                if ret:
                    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
                    current_zone_snap_path = f"{zone_h_p}/zone_{ts}.jpg"
                    snap_draw = cv2.resize(f_snap, (tw, th))
                    cv2.polylines(snap_draw, [zone_pts], True, (0, 255, 255), 2)
                    cv2.imwrite(current_zone_snap_path, snap_draw)
                    last_known_coords[cam_id] = zone_list

            writer, is_recording, rec_start = None, False, 0
            curr_vid_url = ""

            while not stop_now and cap.isOpened():
                ret, frame = cap.read()
                if not ret: break
                frame = cv2.resize(frame, (tw, th))
                raw_frame, curr_time = frame.copy(), time.time()

                with gpu_lock:
                    # FIXED: Corrected 'imgsz' parameter
                    results = MODEL.track(frame, persist=True, imgsz=640, verbose=False, device=DEVICE, classes=[0])

                annotated = frame.copy()
                cv2.polylines(annotated, [zone_pts], True, (0, 255, 255), 2)

                active_ids = []
                if results[0].boxes.id is not None:
                    boxes, ids = results[0].boxes.xyxy.cpu().numpy(), results[0].boxes.id.cpu().numpy().astype(int)
                    for box, pid in zip(boxes, ids):
                        x1, y1, x2, y2 = map(int, box)
                        
                        if cv2.pointPolygonTest(zone_pts, (int((x1+x2)/2), y2), False) >= 0:
                            active_ids.append(pid)
                            
                            # --- VIDEO ID LABEL (Annotated Frame Only) ---
                            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 0, 255), 2)
                            cv2.putText(annotated, f"ID:{pid}", (x1, y1-10), 
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

                            if pid not in tracking_db[cam_id]:
                                # --- WIDE & CLEAN CAPTURE (From raw_frame) ---
                                bw, bh = x2 - x1, y2 - y1
                                pad_w, pad_h = int(bw * 0.45), int(bh * 0.45) # 45% Padding (Door se)
                                zx1, zy1 = max(0, x1 - pad_w), max(0, y1 - pad_h)
                                zx2, zy2 = min(tw, x2 + pad_w), min(th, y2 + pad_h)
                                
                                clean_wide_img = raw_frame[zy1:zy2, zx1:zx2]
                                img_path = f"{caps_p}/intruder_{pid}.jpg"
                                cv2.imwrite(img_path, clean_wide_img)
                                
                                entry_data = {
                                    "person_id": int(pid), "camera": name,
                                    "time_in": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                                    "intruder_img_url": img_path,
                                    "zone_coords": zone_list,
                                    "zone_snap_url": current_zone_snap_path,
                                    "_last_seen": curr_time
                                }
                                tracking_db[cam_id][pid] = entry_data
                                sync_to_openremote(entry_data, asset_id, name)
                                print(f" [ALERT] {name}: Clean Wide Capture saved for ID {pid}")

                                if not is_recording:
                                    is_recording, rec_start = True, curr_time
                                    curr_vid_url = f"{vids_p}/{name}_Clip_{part_counters[cam_id]}.avi"
                                    writer = cv2.VideoWriter(curr_vid_url, cv2.VideoWriter_fourcc(*'XVID'), FPS, (tw*2, th))
                            else:
                                tracking_db[cam_id][pid]["_last_seen"] = curr_time

                combined = np.hstack((raw_frame, annotated))
                if is_recording and writer:
                    writer.write(combined)
                    if (curr_time - rec_start) >= CLIP_DURATION:
                        writer.release()
                        is_recording = False
                        for pid in tracking_db[cam_id]:
                            if "video_url" not in tracking_db[cam_id][pid]:
                                tracking_db[cam_id][pid]["video_url"] = curr_vid_url
                        part_counters[cam_id] += 1

                for pid in list(tracking_db[cam_id].keys()):
                    if pid not in active_ids and (curr_time - tracking_db[cam_id][pid]["_last_seen"] > EXIT_THRESHOLD):
                        tracking_db[cam_id][pid]["time_out"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                        tracking_db[cam_id][pid]["video_url"] = curr_vid_url if curr_vid_url else "N/A"
                        sync_to_openremote(tracking_db[cam_id][pid], asset_id, name)
                        del tracking_db[cam_id][pid]

            cap.release()
            if writer: writer.release()
        except: time.sleep(5)

if __name__ == "__main__":
    for cid in CAMS_DATA.keys():
        threading.Thread(target=process_camera, args=(cid,), daemon=True).start()
    while not stop_now:
        try: time.sleep(1)
        except KeyboardInterrupt: stop_now = True